# 08_3_6 DQA Scene Phase2 SCoLQ Policy

`01_train_and_select_scolq.ipynb` の結果を Phase2 に入れる実験。SCoLQ = **Source-Calibrated Localization Quality Judge** を使い、teacher confidence ではなく source GT で校正した bbox localization quality を pseudo label の gate score にする。

今回の読みはかなり明確で、生 pseudo box は source_val で `good50=9.1%` しかない一方、SCoLQ 上位は `top 5% good50=87.9%`, `top 10% good50=65.8%`, `top 20% good50=40.7%` まで濃縮できていた。なので 08_3_6 では、bbox regression を SCoLQ 高スコア box に限定し、低スコア box は objectness の弱い教師としてだけ残す。

注意点として、`rider/bus/bike/motor/train` は class 別診断で弱い。variant C ではこれらの class の bbox gate を高くして、rare/weak class の localization drift を避ける。

In [1]:
from __future__ import annotations

import json
import os
import signal
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "dynamic_quality_aware_classwise_aggregation").exists() and (path / "navigating_data_heterogeneity").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2_phase2_scolq_policy.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_scene_protocol.py"
SOURCE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_scene_tri_stage_policy_8h"
SCOLQ_ROOT = DQA_ROOT / "source_calibrated_localization_quality"
SCOLQ_MODEL = SCOLQ_ROOT / "artifacts" / "scolq_best.joblib"
SCOLQ_RANKING = SCOLQ_ROOT / "reports" / "model_ranking.csv"
SCOLQ_DIAG = SCOLQ_ROOT / "reports" / "best_model_class_diagnostics.csv"
BASE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_3_6_phase2_scolq_policy"
BASE_STATS_ROOT = DQA_ROOT / "stats_dqa08_3_6_phase2_scolq_policy"
BASE_LOG_ROOT = DQA_ROOT / "logs_dqa08_3_6_phase2_scolq_policy"

for path in [RUN_SCRIPT, EVAL_SCRIPT, SOURCE_WORK_ROOT, SCOLQ_MODEL]:
    print(path, "exists=", path.exists())

BASE_WORK_ROOT.mkdir(parents=True, exist_ok=True)
BASE_STATS_ROOT.mkdir(parents=True, exist_ok=True)
BASE_LOG_ROOT.mkdir(parents=True, exist_ok=True)

if SCOLQ_RANKING.exists():
    ranking = pd.read_csv(SCOLQ_RANKING)
    display(ranking.head(8)[["candidate", "n_features", "ap50_quality", "ap75_quality", "p50_at_10pct", "p50_at_20pct", "selection_score"]])
if SCOLQ_DIAG.exists():
    diag = pd.read_csv(SCOLQ_DIAG)
    display(diag[["class_name", "base_good50_rate", "p50_at_10pct", "p50_at_20pct"]])

print("workspace:", BASE_WORK_ROOT)
print("stats:", BASE_STATS_ROOT)
print("logs:", BASE_LOG_ROOT)

/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase2_scolq_policy.py exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h exists= True
/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/source_calibrated_localization_quality/artifacts/scolq_best.joblib exists= True


,candidate,n_features,ap50_quality,ap75_quality,p50_at_10pct,p50_at_20pct,selection_score
0,all:hgb,28,0.773647,0.792086,0.658092,0.406892,1.459251
1,score_geometry_context:hgb,20,0.773349,0.792675,0.657626,0.406589,1.459128
2,score_geometry_context_prior:hgb,24,0.773440,0.792300,0.657253,0.406706,1.459047
3,score_geometry_context:rf,20,0.772642,0.793938,0.657067,0.407172,1.453971
4,score_geometry_context_prior:rf,24,0.772501,0.793847,0.656648,0.407102,1.453667
5,all:rf,28,0.772693,0.792711,0.656741,0.407102,1.453379
6,score_context:hgb,11,0.766983,0.795488,0.652221,0.404446,1.451941
7,score_context:rf,11,0.764188,0.795532,0.650217,0.403770,1.441428


,class_name,base_good50_rate,p50_at_10pct,p50_at_20pct
0,person,0.055407,0.430025,0.251538
1,rider,0.005706,0.051311,0.026811
2,car,0.207672,0.948869,0.756280
3,bus,0.013712,0.104076,0.060304
4,truck,0.037225,0.273678,0.162996
5,bike,0.009788,0.086022,0.046774
6,motor,0.007274,0.061818,0.034545
7,traffic light,0.084312,0.590643,0.373094
8,traffic sign,0.074931,0.533041,0.321589
9,train,0.000143,0.000000,0.000000


workspace: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_6_phase2_scolq_policy
stats: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_3_6_phase2_scolq_policy
logs: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_6_phase2_scolq_policy


## Variant Design

SCoLQ の source_val threshold sweep では `scolq>=0.60` が coverage `5.5%` で `good50=85.2%`、`scolq>=0.70` が coverage `4.6%` で `good50=89.7%` だった。bbox regression に流すなら `0.60-0.70` が現実的。

- A: まずの本命。`high=0.60` で SCoLQ 高品質 box だけ bbox/cls positive、残りは弱 objectness。
- B: precision 優先。`high=0.70`、pseudo 数も絞る。
- C: class-wise guard。SCoLQ 診断で弱かった `rider/bus/bike/motor/train` の bbox gate を上げる。
- D: bbox をほぼ使わず、SCoLQ を DQA quality と objectness だけに使う対照。

In [2]:
PHASE2_ROUNDS_PER_VARIANT = 10
BATCH_SIZE = 160
WORKERS = 10
GPUS = 2
MASTER_PORT_BASE = 29960
STREAM_TRAIN_OUTPUT = False
MIN_FREE_GIB = 30

# まずは本命Aだけ回す。比較を増やすなら [] にすると全variant、または C/B/D を追加する。
SELECTED_VARIANTS: list[str] = ["a_scolq_soft_bbox_r003"]

CLASSWISE_WEAK_GUARD_HIGH = [0.60, 0.80, 0.55, 0.80, 0.65, 0.80, 0.80, 0.60, 0.60, 0.90]

VARIANTS = [
    {
        "name": "a_scolq_soft_bbox_r003",
        "description": "SCoLQ>=0.60だけをbbox/cls positiveにする本命。低SCoLQは弱objectnessとして残す。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "all",
        "classwise_blend": 0.065,
        "server_anchor": 13.0,
        "temperature": 2.7,
        "stability_lambda": 0.66,
        "residual_start": 0.14,
        "residual_end": 0.06,
        "min_server_alpha_start": 0.70,
        "min_server_alpha_end": 0.74,
        "env": {
            "DQA0836_SCOLQ_LOW": 0.10,
            "DQA0836_SCOLQ_MID": 0.30,
            "DQA0836_SCOLQ_HIGH": 0.60,
            "DQA0836_NMS_CONF_THRES": 0.01,
            "DQA0836_TEACHER_LOSS_WEIGHT": 0.32,
            "DQA0836_BOX_LOSS_WEIGHT": 0.010,
            "DQA0836_OBJ_LOSS_WEIGHT": 0.32,
            "DQA0836_CLS_LOSS_WEIGHT": 0.08,
            "DQA0836_LOW_MID_OBJ_WEIGHT": 0.25,
            "DQA0836_MID_HIGH_OBJ_WEIGHT": 0.70,
            "DQA0836_CLIENT_LR0_START": 0.0010,
            "DQA0836_CLIENT_LR0_END": 0.00022,
            "DQA0836_SERVER_LR0_START": 0.0030,
            "DQA0836_SERVER_LR0_END": 0.0010,
            "DQA0836_MAX_PSEUDO_PER_IMAGE": 80,
            "DQA0836_MAX_PSEUDO_PER_CLASS_IMAGE": 25,
        },
    },
    {
        "name": "b_scolq_strict_bbox_r003",
        "description": "SCoLQ>=0.70だけbbox positive。bboxはかなり高精度に寄せ、pseudo数も絞る。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "all",
        "classwise_blend": 0.050,
        "server_anchor": 16.0,
        "temperature": 3.0,
        "stability_lambda": 0.72,
        "residual_start": 0.10,
        "residual_end": 0.03,
        "min_server_alpha_start": 0.76,
        "min_server_alpha_end": 0.84,
        "env": {
            "DQA0836_SCOLQ_LOW": 0.15,
            "DQA0836_SCOLQ_MID": 0.40,
            "DQA0836_SCOLQ_HIGH": 0.70,
            "DQA0836_NMS_CONF_THRES": 0.015,
            "DQA0836_TEACHER_LOSS_WEIGHT": 0.28,
            "DQA0836_BOX_LOSS_WEIGHT": 0.008,
            "DQA0836_OBJ_LOSS_WEIGHT": 0.28,
            "DQA0836_CLS_LOSS_WEIGHT": 0.05,
            "DQA0836_LOW_MID_OBJ_WEIGHT": 0.18,
            "DQA0836_MID_HIGH_OBJ_WEIGHT": 0.55,
            "DQA0836_CLIENT_LR0_START": 0.0008,
            "DQA0836_CLIENT_LR0_END": 0.00016,
            "DQA0836_SERVER_LR0_START": 0.0030,
            "DQA0836_SERVER_LR0_END": 0.0009,
            "DQA0836_MAX_PSEUDO_PER_IMAGE": 55,
            "DQA0836_MAX_PSEUDO_PER_CLASS_IMAGE": 16,
        },
    },
    {
        "name": "c_scolq_classwise_weak_guard_r003",
        "description": "弱いclassのbbox gateを上げる。rare classの誤bbox regressionで全体が崩れるのを避ける。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "neck_head",
        "classwise_blend": 0.060,
        "server_anchor": 14.0,
        "temperature": 2.8,
        "stability_lambda": 0.68,
        "residual_start": 0.12,
        "residual_end": 0.04,
        "min_server_alpha_start": 0.74,
        "min_server_alpha_end": 0.82,
        "env": {
            "DQA0836_SCOLQ_LOW": 0.10,
            "DQA0836_SCOLQ_MID": 0.30,
            "DQA0836_CLASSWISE_HIGH": json.dumps(CLASSWISE_WEAK_GUARD_HIGH),
            "DQA0836_NMS_CONF_THRES": 0.01,
            "DQA0836_TEACHER_LOSS_WEIGHT": 0.30,
            "DQA0836_BOX_LOSS_WEIGHT": 0.010,
            "DQA0836_OBJ_LOSS_WEIGHT": 0.32,
            "DQA0836_CLS_LOSS_WEIGHT": 0.06,
            "DQA0836_LOW_MID_OBJ_WEIGHT": 0.22,
            "DQA0836_MID_HIGH_OBJ_WEIGHT": 0.65,
            "DQA0836_CLIENT_LR0_START": 0.0009,
            "DQA0836_CLIENT_LR0_END": 0.00018,
            "DQA0836_SERVER_LR0_START": 0.0030,
            "DQA0836_SERVER_LR0_END": 0.0010,
            "DQA0836_MAX_PSEUDO_PER_IMAGE": 80,
            "DQA0836_MAX_PSEUDO_PER_CLASS_IMAGE": 20,
        },
    },
    {
        "name": "d_scolq_obj_only_dqa_r003",
        "description": "bbox regressionをほぼ切る対照。SCoLQをDQA qualityと弱objectnessだけに使う。",
        "source_phase1_round": 3,
        "dqa_start_phase": 2,
        "client_train_scope": "neck_head",
        "classwise_blend": 0.055,
        "server_anchor": 15.0,
        "temperature": 2.8,
        "stability_lambda": 0.70,
        "residual_start": 0.08,
        "residual_end": 0.02,
        "min_server_alpha_start": 0.78,
        "min_server_alpha_end": 0.88,
        "env": {
            "DQA0836_SCOLQ_LOW": 0.10,
            "DQA0836_SCOLQ_MID": 0.35,
            "DQA0836_SCOLQ_HIGH": 0.95,
            "DQA0836_NMS_CONF_THRES": 0.01,
            "DQA0836_TEACHER_LOSS_WEIGHT": 0.30,
            "DQA0836_BOX_LOSS_WEIGHT": 0.000,
            "DQA0836_OBJ_LOSS_WEIGHT": 0.34,
            "DQA0836_CLS_LOSS_WEIGHT": 0.00,
            "DQA0836_LOW_MID_OBJ_WEIGHT": 0.25,
            "DQA0836_MID_HIGH_OBJ_WEIGHT": 0.60,
            "DQA0836_CLIENT_LR0_START": 0.0008,
            "DQA0836_CLIENT_LR0_END": 0.00016,
            "DQA0836_SERVER_LR0_START": 0.0030,
            "DQA0836_SERVER_LR0_END": 0.0010,
            "DQA0836_MAX_PSEUDO_PER_IMAGE": 80,
            "DQA0836_MAX_PSEUDO_PER_CLASS_IMAGE": 25,
        },
    },
]

selected = [v for v in VARIANTS if not SELECTED_VARIANTS or v["name"] in SELECTED_VARIANTS]
print("selected:", [v["name"] for v in selected])

selected: ['a_scolq_soft_bbox_r003']


In [3]:
def variant_work_root(variant: dict) -> Path:
    return BASE_WORK_ROOT / variant["name"]


def variant_stats_root(variant: dict) -> Path:
    return BASE_STATS_ROOT / variant["name"]


def variant_runner_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_runner.out"


def variant_train_log(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}_train.log"


def variant_pid_path(variant: dict) -> Path:
    return BASE_LOG_ROOT / f"{variant['name']}.pid"


def variant_env(variant: dict) -> dict[str, str]:
    env = os.environ.copy()
    stats_root = variant_stats_root(variant)
    stats_root.mkdir(parents=True, exist_ok=True)
    env["DQA0836_VARIANT"] = variant["name"]
    env["DQA0836_SOURCE_WORK_ROOT"] = str(SOURCE_WORK_ROOT)
    env["DQA0836_SOURCE_PHASE1_ROUND"] = str(variant["source_phase1_round"])
    env["DQA0836_SCOLQ_MODEL"] = str(SCOLQ_MODEL)
    env["DQA0836_CLIENT_TRAIN_SCOPE"] = variant["client_train_scope"]
    env["DQA0836_SERVER_TRAIN_SCOPE"] = "all"
    env["DQA0836_RESIDUAL_START"] = str(variant["residual_start"])
    env["DQA0836_RESIDUAL_END"] = str(variant["residual_end"])
    env["DQA0836_MIN_SERVER_ALPHA_START"] = str(variant["min_server_alpha_start"])
    env["DQA0836_MIN_SERVER_ALPHA_END"] = str(variant["min_server_alpha_end"])
    env["DQA0836_PHASE2_ROUNDS"] = str(PHASE2_ROUNDS_PER_VARIANT)
    env["DQA08_STATS_ROOT"] = str(stats_root.resolve())
    env["DQA08_THRESHOLD_LOG"] = str((stats_root / "phase2_scolq_policy_schedule.jsonl").resolve())
    env["DQA_STATS_QUALITY_MODE"] = "scolq"
    env["DQA0834_STATS_QUALITY_MODE"] = "scolq"
    env.setdefault("ET_SKIP_AFTER_TRAIN_BEST_VAL", "1")
    for key, value in variant.get("env", {}).items():
        env[key] = str(value)
    return env


def train_cmd(variant: dict, *, stream: bool = STREAM_TRAIN_OUTPUT) -> list[str]:
    cmd = [
        sys.executable,
        str(RUN_SCRIPT),
        "--workspace-root",
        str(variant_work_root(variant)),
        "--stats-root",
        str(variant_stats_root(variant)),
        "--phase1-rounds",
        "0",
        "--phase2-rounds",
        str(PHASE2_ROUNDS_PER_VARIANT),
        "--batch-size",
        str(BATCH_SIZE),
        "--workers",
        str(WORKERS),
        "--gpus",
        str(GPUS),
        "--master-port",
        str(MASTER_PORT_BASE + selected.index(variant)),
        "--log-file",
        str(variant_train_log(variant)),
        "--classwise-blend",
        str(variant["classwise_blend"]),
        "--server-anchor",
        str(variant["server_anchor"]),
        "--temperature",
        str(variant["temperature"]),
        "--stability-lambda",
        str(variant["stability_lambda"]),
        "--dqa-start-phase",
        str(variant["dqa_start_phase"]),
        "--min-free-gib",
        str(MIN_FREE_GIB),
    ]
    if stream:
        cmd.append("--stream-train-output")
    return cmd


def history_rows(variant: dict) -> list[dict]:
    path = variant_work_root(variant) / "history.json"
    if not path.exists():
        return []
    return json.loads(path.read_text())


def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    try:
        return int(path.read_text().strip())
    except Exception:
        return None


def pid_state(pid: int | None) -> str:
    if pid is None:
        return "no pid"
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        return "stopped"
    except PermissionError:
        return "running?"
    return "running"


def tail_lines(path: Path, n: int = 30) -> list[str]:
    if not path.exists():
        return [f"missing: {path}"]
    return path.read_text(errors="replace").splitlines()[-n:]

print("helpers ready")

helpers ready


In [4]:
# 実行セル。途中で止まっても history/global checkpoint があれば再利用します。
if not selected:
    raise RuntimeError("No selected variants")
if not SCOLQ_MODEL.exists():
    raise FileNotFoundError(f"Missing SCoLQ model: {SCOLQ_MODEL}")

for variant in selected:
    history = history_rows(variant)
    completed_phase2 = sum(1 for row in history if int(row.get("phase", 0)) == 2)
    print("\n===", variant["name"], "===")
    print(variant["description"])
    print(f"completed_phase2: {completed_phase2}/{PHASE2_ROUNDS_PER_VARIANT}")
    if completed_phase2 >= PHASE2_ROUNDS_PER_VARIANT:
        print("already completed")
        continue

    pid_path = variant_pid_path(variant)
    existing_pid = read_pid(pid_path)
    if pid_state(existing_pid) == "running":
        print("already running pid:", existing_pid)
        continue

    runner_log = variant_runner_log(variant)
    runner_log.parent.mkdir(parents=True, exist_ok=True)
    cmd = train_cmd(variant)
    env = variant_env(variant)
    print("cmd:", " ".join(cmd))
    print("runner_log:", runner_log)
    print("train_log:", variant_train_log(variant))

    with runner_log.open("a", encoding="utf-8", buffering=1) as log:
        log.write("\n" + "=" * 100 + "\n")
        log.write(" ".join(cmd) + "\n")
        process = subprocess.Popen(
            cmd,
            cwd=REPO_ROOT,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        pid_path.write_text(str(process.pid), encoding="utf-8")
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        rc = process.wait()
        if rc != 0:
            raise RuntimeError(f"{variant['name']} failed with exit code {rc}. See {runner_log}")
        print("variant completed:", variant["name"])


=== a_scolq_soft_bbox_r003 ===
SCoLQ>=0.60だけをbbox/cls positiveにする本命。低SCoLQは弱objectnessとして残す。
completed_phase2: 0/10
cmd: /root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase2_scolq_policy.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_6_phase2_scolq_policy/a_scolq_soft_bbox_r003 --stats-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_3_6_phase2_scolq_policy/a_scolq_soft_bbox_r003 --phase1-rounds 0 --phase2-rounds 10 --batch-size 160 --workers 10 --gpus 2 --master-port 29960 --log-file /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_6_phase2_scolq_policy/a_scolq_soft_bbox_r003_train.log --classwise-blend 0.065 --server-anchor 13.0 --temperature 2.7 --stability-lambda 0.66 --dqa-start-phase 2 --min-free-gib 30
runner_log: /app/Object_Detection/dynamic_quality_aware_class

In [5]:
# 進捗確認セル
rows = []
for variant in selected:
    history = history_rows(variant)
    latest = Path(history[-1]["global"]) if history else variant_work_root(variant) / "global_checkpoints" / "round000_warmup.pt"
    rows.append({
        "variant": variant["name"],
        "pid": read_pid(variant_pid_path(variant)),
        "pid_state": pid_state(read_pid(variant_pid_path(variant))),
        "phase2": f"{sum(1 for row in history if int(row.get('phase', 0)) == 2)}/{PHASE2_ROUNDS_PER_VARIANT}",
        "latest": str(latest),
        "latest_exists": latest.exists(),
        "free_gib": round(shutil.disk_usage(variant_work_root(variant)).free / 1024**3, 2) if variant_work_root(variant).exists() else None,
    })

display(pd.DataFrame(rows))

for variant in selected:
    print("\n===", variant["name"], "runner tail ===")
    print("\n".join(tail_lines(variant_runner_log(variant), 24)))

,variant,pid,pid_state,phase2,latest,latest_exists,free_gib
0,a_scolq_soft_bbox_r003,1342656,stopped,10/10,/app/Object_Detection/dynamic_quality_aware_cl...,True,82.19



=== a_scolq_soft_bbox_r003 runner tail ===
Training output appended to /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_6_phase2_scolq_policy/a_scolq_soft_bbox_r003_train.log
DQA08_3_6 SCoLQ gate: client=1 round=9 low=0.10 mid=0.30 high=0.60 model=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/source_calibrated_localization_quality/artifacts/scolq_best.joblib
/root/micromamba/envs/al_yolov8/bin/python -m torch.distributed.run --nproc_per_node 2 --master_port 29960 train.py --cfg /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_6_phase2_scolq_policy/a_scolq_soft_bbox_r003/configs/runtime_phase2_client_round9_dqa_phase2_round009_client1_citystreet.yaml
Training output appended to /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/logs_dqa08_3_6_phase2_scolq_policy/a_scolq_soft_bbox_r003_train.log
DQA08_3_6 SCoLQ gate: client=2 round=9 low=0.10 mid=0.30 high=0.60 model=/app/Object_De

In [6]:
# early/final checkpoint 評価。Phase2 は early peak しやすいので r001/r002/r003/r010 を見る。
EVAL_WORKSPACE = variant_work_root(selected[0])
REPORT_DIR = EVAL_WORKSPACE / "validation_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

checkpoints: list[tuple[str, Path]] = []
seed03 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round003_global.pt"
seed12 = SOURCE_WORK_ROOT / "global_checkpoints" / "phase1_round012_global.pt"
old0834_c02 = DQA_ROOT / "efficientteacher_dqa08_3_4_phase2_feature_quality_sweep" / "c_feature_balanced_neck_head_r003" / "global_checkpoints" / "phase2_round002_global.pt"
old0834_d02 = DQA_ROOT / "efficientteacher_dqa08_3_4_phase2_feature_quality_sweep" / "d_feature_conservative_min_gate_r003" / "global_checkpoints" / "phase2_round002_global.pt"
checkpoints.extend([
    ("p1_r003", seed03),
    ("p1_r012", seed12),
    ("old08_3_4_c_r002", old0834_c02),
    ("old08_3_4_d_r002", old0834_d02),
])

for variant in selected:
    root = variant_work_root(variant) / "global_checkpoints"
    for r in [1, 2, 3, PHASE2_ROUNDS_PER_VARIANT]:
        ckpt = root / f"phase2_round{r:03d}_global.pt"
        if ckpt.exists():
            checkpoints.append((f"{variant['name']}_r{r:03d}", ckpt))

print("eval_workspace:", EVAL_WORKSPACE)
for label, path in checkpoints:
    print(label, path, "exists=", path.exists())

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--workspace",
    str(EVAL_WORKSPACE),
    "--splits",
    "total",
    "--batch-size",
    "16",
    "--no-plots",
    "--verbose",
]
for label, path in checkpoints:
    if path.exists():
        cmd.extend(["--checkpoint", f"{label}={path}"])

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

summary_csv = REPORT_DIR / "paper_protocol_eval_summary.csv"
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    out = REPORT_DIR / "paper_protocol_eval_summary_0836_early_total.csv"
    df.to_csv(out, index=False)
    display(df.sort_values(["split", "map50_95", "map50"], ascending=[True, False, False])[["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]])
    print("saved:", out)
else:
    print("No summary yet:", summary_csv)

eval_workspace: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_6_phase2_scolq_policy/a_scolq_soft_bbox_r003
p1_r003 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round003_global.pt exists= True
p1_r012 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round012_global.pt exists= True
old08_3_4_c_r002 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_4_phase2_feature_quality_sweep/c_feature_balanced_neck_head_r003/global_checkpoints/phase2_round002_global.pt exists= True
old08_3_4_d_r002 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_4_phase2_feature_quality_sweep/d_feature_conservative_min_gate_r003/global_checkpoints/phase2_round002_global.pt exists= True
a_scolq_soft_bbox_r00

,checkpoint_label,split,precision,recall,map50,map50_95
3,old08_3_4_d_r002,scene_total,0.503,0.399,0.381,0.213
5,a_scolq_soft_bbox_r003_r002,scene_total,0.489,0.411,0.381,0.212
2,old08_3_4_c_r002,scene_total,0.492,0.408,0.380,0.212
4,a_scolq_soft_bbox_r003_r001,scene_total,0.493,0.401,0.382,0.211
6,a_scolq_soft_bbox_r003_r003,scene_total,0.517,0.388,0.377,0.211
0,p1_r003,scene_total,0.471,0.401,0.375,0.204
1,p1_r012,scene_total,0.743,0.326,0.368,0.199
7,a_scolq_soft_bbox_r003_r010,scene_total,0.483,0.389,0.357,0.196


saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_6_phase2_scolq_policy/a_scolq_soft_bbox_r003/validation_reports/paper_protocol_eval_summary_0836_early_total.csv


In [7]:
# 4 split final evaluation。上の total 評価で良い checkpoint を BEST_LABELS に入れる。
BEST_LABELS = []  # 例: ["a_scolq_soft_bbox_r003_r001", "c_scolq_classwise_weak_guard_r003_r002"]

ckpt_map = {label: path for label, path in checkpoints if path.exists()}
if not BEST_LABELS:
    BEST_LABELS = [
        label for label in ckpt_map
        if label.endswith("_r001") or label.endswith("_r002") or label.endswith(f"r{PHASE2_ROUNDS_PER_VARIANT:03d}")
    ]

cmd = [
    sys.executable,
    str(EVAL_SCRIPT),
    "--workspace",
    str(EVAL_WORKSPACE),
    "--splits",
    "highway,citystreet,residential,total",
    "--batch-size",
    "16",
    "--no-plots",
    "--verbose",
]
for label in ["p1_r003", "p1_r012", "old08_3_4_c_r002", "old08_3_4_d_r002", *BEST_LABELS]:
    path = ckpt_map.get(label)
    if path and path.exists():
        cmd.extend(["--checkpoint", f"{label}={path}"])

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

summary_csv = REPORT_DIR / "paper_protocol_eval_summary.csv"
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    out = REPORT_DIR / "paper_protocol_eval_summary_0836_selected_4splits.csv"
    df.to_csv(out, index=False)
    display(df.sort_values(["split", "map50_95", "map50"], ascending=[True, False, False])[["checkpoint_label", "split", "precision", "recall", "map50", "map50_95"]])
    print("saved:", out)

/root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py --workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_6_phase2_scolq_policy/a_scolq_soft_bbox_r003 --splits highway,citystreet,residential,total --batch-size 16 --no-plots --verbose --checkpoint p1_r003=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round003_global.pt --checkpoint p1_r012=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/global_checkpoints/phase1_round012_global.pt --checkpoint old08_3_4_c_r002=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_4_phase2_feature_quality_sweep/c_feature_balanced_neck_head_r003/global_checkpoints/phase2_round002_global.pt --checkpoint old08_3_4_d_r002=/app/Objec

,checkpoint_label,split,precision,recall,map50,map50_95
13,old08_3_4_d_r002,citystreet,0.488,0.418,0.387,0.216
21,old08_3_4_d_r002,citystreet,0.488,0.418,0.387,0.216
29,a_scolq_soft_bbox_r003_r002,citystreet,0.488,0.418,0.387,0.216
9,old08_3_4_c_r002,citystreet,0.493,0.414,0.386,0.216
17,old08_3_4_c_r002,citystreet,0.493,0.414,0.386,0.216
25,a_scolq_soft_bbox_r003_r001,citystreet,0.484,0.414,0.387,0.215
1,p1_r003,citystreet,0.472,0.409,0.381,0.207
5,p1_r012,citystreet,0.745,0.332,0.374,0.202
33,a_scolq_soft_bbox_r003_r010,citystreet,0.479,0.399,0.364,0.199
28,a_scolq_soft_bbox_r003_r002,highway,0.478,0.368,0.333,0.188


saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_6_phase2_scolq_policy/a_scolq_soft_bbox_r003/validation_reports/paper_protocol_eval_summary_0836_selected_4splits.csv


In [8]:
# pseudoGT / SCoLQ quality / DQA gate 推移を確認するセル。
rows = []
for variant in selected:
    stats_root = variant_stats_root(variant)
    for r in range(1, PHASE2_ROUNDS_PER_VARIANT + 1):
        path = stats_root / f"phase2_round{r:03d}.json"
        if not path.exists():
            continue
        data = json.loads(path.read_text())
        counts = [0.0] * 10
        qsum = [0.0] * 10
        confsum = [0.0] * 10
        objsum = [0.0] * 10
        for client in data.get("clients", []):
            for i, value in enumerate(client.get("counts", [])):
                counts[i] += float(value)
            for i, value in enumerate(client.get("quality_sums", [])):
                qsum[i] += float(value)
            for i, value in enumerate(client.get("confidence_sums", [])):
                confsum[i] += float(value)
            for i, value in enumerate(client.get("objectness_sums", [])):
                objsum[i] += float(value)
        total = sum(counts)
        rows.append({
            "variant": variant["name"],
            "round": r,
            "pseudo_total": total,
            "mean_scolq_quality": sum(qsum) / total if total else None,
            "mean_score_column": sum(confsum) / total if total else None,
            "mean_objectness": sum(objsum) / total if total else None,
            "active_classes": sum(1 for x in counts if x > 0),
            "person_count": counts[0],
            "car_count": counts[2],
            "traffic_light_count": counts[7],
            "traffic_sign_count": counts[8],
        })

stats_df = pd.DataFrame(rows)
if not stats_df.empty:
    display(stats_df)
    display(stats_df.groupby("variant")[["pseudo_total", "mean_scolq_quality", "mean_objectness", "active_classes"]].agg(["min", "max", "mean"]))
else:
    print("No pseudo stats yet")

for variant in selected:
    state_path = variant_work_root(variant) / "dqa_cwa_state.json"
    if not state_path.exists():
        continue
    state = json.loads(state_path.read_text())
    gate = state.get("phase2_scolq_policy", [])
    if gate:
        print()
        print(variant["name"])
        display(pd.DataFrame(gate))

,variant,round,pseudo_total,mean_scolq_quality,mean_score_column,mean_objectness,active_classes,person_count,car_count,traffic_light_count,traffic_sign_count
0,a_scolq_soft_bbox_r003,1,770515.0,0.443193,0.443193,0.344575,9,86931.0,316177.0,103519.0,230143.0
1,a_scolq_soft_bbox_r003,2,773621.0,0.448643,0.448643,0.353537,8,81923.0,316237.0,103402.0,235124.0
2,a_scolq_soft_bbox_r003,3,757997.0,0.450761,0.450761,0.353734,8,87371.0,316108.0,100409.0,221301.0
3,a_scolq_soft_bbox_r003,4,760731.0,0.453214,0.453214,0.355148,8,83274.0,315941.0,103382.0,226415.0
4,a_scolq_soft_bbox_r003,5,770963.0,0.456014,0.456014,0.357044,8,88444.0,315865.0,104628.0,229311.0
5,a_scolq_soft_bbox_r003,6,785763.0,0.460806,0.460806,0.360953,8,88819.0,316050.0,110817.0,236926.0
6,a_scolq_soft_bbox_r003,7,795512.0,0.466490,0.466490,0.365772,8,90509.0,316491.0,112750.0,241664.0
7,a_scolq_soft_bbox_r003,8,807088.0,0.470166,0.470166,0.369221,8,92436.0,316008.0,113220.0,249392.0
8,a_scolq_soft_bbox_r003,9,817731.0,0.476864,0.476864,0.375695,9,94073.0,316611.0,117122.0,252518.0
9,a_scolq_soft_bbox_r003,10,819653.0,0.487045,0.487045,0.385868,9,96931.0,316338.0,114717.0,253150.0


pseudo_total                     mean_scolq_quality  \
                                min       max      mean                min   
variant                                                                      
a_scolq_soft_bbox_r003     757997.0  819653.0  785957.4           0.443193   

                                          mean_objectness                      \
                             max     mean             min       max      mean   
variant                                                                         
a_scolq_soft_bbox_r003  0.487045  0.46132        0.344575  0.385868  0.362155   

                       active_classes           
                                  min max mean  
variant                                         
a_scolq_soft_bbox_r003              8   9  8.3


a_scolq_soft_bbox_r003


,classwise_blend,min_server_alpha,residual_blend,round,scolq_high,scolq_low,scolq_mid,scolq_model,server_anchor,variant
0,0.065,0.700000,0.140000,1,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,13.0,a_scolq_soft_bbox_r003
1,0.065,0.704444,0.131111,2,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,13.0,a_scolq_soft_bbox_r003
2,0.065,0.708889,0.122222,3,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,13.0,a_scolq_soft_bbox_r003
3,0.065,0.713333,0.113333,4,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,13.0,a_scolq_soft_bbox_r003
4,0.065,0.717778,0.104444,5,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,13.0,a_scolq_soft_bbox_r003
5,0.065,0.722222,0.095556,6,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,13.0,a_scolq_soft_bbox_r003
6,0.065,0.726667,0.086667,7,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,13.0,a_scolq_soft_bbox_r003
7,0.065,0.731111,0.077778,8,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,13.0,a_scolq_soft_bbox_r003
8,0.065,0.735556,0.068889,9,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,13.0,a_scolq_soft_bbox_r003
9,0.065,0.740000,0.060000,10,"[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, ...",/app/Object_Detection/dynamic_quality_aware_cl...,13.0,a_scolq_soft_bbox_r003


## 読み方

A が `r001/r002` で Phase1 を超えるなら、SCoLQ bbox gate はそのまま採用候補。`r010` まで落ちる場合は、SCoLQ でも毎 round の target bbox regression が強すぎるので、B/C のように high gate を上げるか residual をさらに下げる。

C が A より安定するなら、問題は SCoLQ 全体ではなく class-wise に弱い bbox を流していること。D が良い場合は、Phase2 では bbox regression そのものをほぼ止め、SCoLQ は DQA aggregation と objectness repair のみに使う方がよい。